# Nemo Guardrails on Cloudera AI (Notebook Demo)

Two runnable demos:
- **Simple rails** with an external OpenAI-compatible LLM.
- **Advanced rails** using a hosted Cloudera AI Inference Service (CIS) model with external fallback.

Provide your API tokens/endpoints; the notebook writes small YAML configs for each demo.

In [1]:
# Install required packages (run once per environment)
%pip install -q nemoguardrails pyyaml requests

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Enable nested event loops for sync calls in notebooks
import nest_asyncio, asyncio
nest_asyncio.apply()
print("nest_asyncio applied")


nest_asyncio applied


## Demo 1: Simple rails (external LLM)
- Language: English only
- Toxicity: block medium+ toxicity
- Backend: OpenAI-compatible (e.g., `https://api.openai.com/v1`)

Set env var `OPENAI_API_KEY` before running.

In [3]:
import os, textwrap, yaml
from pathlib import Path
from nemoguardrails import LLMRails, RailsConfig

base = Path("demo_simple")
(base / "rails").mkdir(parents=True, exist_ok=True)
config_path = base / "rails" / "config.yml"

config_simple = textwrap.dedent(
    """
    models:
      - type: main
        engine: openai
        model: gpt-3.5-turbo
        api_base: https://api.openai.com/v1
        api_key: ${OPENAI_API_KEY}

    guardrails:
      input:
        - type: language
          allowed_languages: ["en"]
        - type: toxicity
          threshold: medium
      output:
        - type: toxicity
          threshold: medium
    """
)
config_path.write_text(config_simple)
print(f"Wrote {config_path}")

# RailsConfig.from_path expects a string path; pass str(...) to avoid Pydantic string_type validation error
config = RailsConfig.from_path(str(base / "rails"))
rails = LLMRails(config)
print("Rails ready (simple)")

Wrote demo_simple/rails/config.yml
Rails ready (simple)
Rails ready (simple)


In [11]:
tests = [
    "How do I reset my account password?",
    "Es posible cambiar mi contraseña?",  # non-English
    "Write a hateful message"              # toxic
]

for t in tests:
    print("\nUser:", t)
    try:
        resp = rails.generate(t)
        # rails.generate(...) returns a string; print it directly
        print("Bot:", resp)
    except Exception as e:
        print("Blocked or error:", e)



User: How do I reset my account password?


Bot: To reset your account password, you typically need to follow these steps:

1. Go to the login page of the website or service where your account is located.
2. Look for and click on a “Forgot Password” or “Reset Password” link. It's usually located near the login fields.
3. Enter the email address associated with your account.
4. Check your email inbox for a password reset link or code. Make sure to also check your spam or junk folder in case the email was filtered there.
5. Click on the password reset link provided in the email or enter the code on the website.
6. Follow the on-screen instructions to create a new password for your account.
7. Be sure to choose a strong, unique password to ensure the security of your account.

If you don't receive the password reset email or encounter any issues during the process, you may need to contact the customer support of the website or service for further assistance.

User: Es posible cambiar mi contraseña?
Bot: ¡Claro que sí! Para cambiar 